In [ ]:
import pandas as pd 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

In [ ]:
cols = ['price', 'year', 'manufacturer', 'model', 'condition', 'cylinders',
        'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'state']
data = pd.read_csv('C:/VsCodeProjects/Used_Cars/vehicles.csv', usecols=cols)

print(data.describe().round().astype(int))
print("Original data: ", data.shape)

In [ ]:
# Cleaning
data = data[data['price'].between(1000, 100000)]
data = data[data['year'].between(1990, 2022)]
data = data[data['odometer'].between(100, 400000)]

data = data.drop_duplicates(subset=['price', 'year', 'manufacturer', 'model', 'odometer'])

for col in ['manufacturer', 'title_status', 'type', 'condition']:
    data[col] = data[col].fillna('unknown')

data['cylinders'] = pd.to_numeric(data['cylinders'].str.split().str[0], errors='coerce')
data['cylinders'] = data['cylinders'].fillna(data['cylinders'].median())

data = data.dropna(subset=['fuel', 'transmission'])

print("After cleaning:", data.shape)
print(data[['price', 'year', 'odometer']].describe())

In [ ]:
features = ['year', 'odometer', 'cylinders', 'fuel', 'transmission', 'manufacturer', 'title_status', 'type']
X = pd.get_dummies(data[features], columns=['fuel', 'transmission', 'manufacturer', 'title_status', 'type'])
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Linear Regression

lr_model = LinearRegression()
lr_model.fit(X_train,y_train)
lr_mae = mean_absolute_error(y_test, lr_model.predict(X_test))
print('Linear Regression MAE: ', lr_mae)

# Naive MAE

naive_pred = np.full(len(y_test), y_train.median())
naive_mae = mean_absolute_error(y_test, naive_pred)
print('Naive MAE: ', naive_mae)

In [ ]:
# Random Forest Regressor 

rf_model = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)
rf_mae = mean_absolute_error(y_test, rf_model.predict(X_test))
print('Random Forest Regressor: ', rf_mae)

imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(10))

In [ ]:
# Hist Gradient Boosting Regressor

hgb = HistGradientBoostingRegressor(loss='absolute_error', max_iter=300, random_state=42)
hgb.fit(X_train, y_train)
hgb_mae = mean_absolute_error(y_test, hgb.predict(X_test))
print("Gradient boosting MAE:", hgb_mae)

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive (median)', 'Linear regression', 'Random forest', 'Gradient boosting'],
    'MAE ($)': [naive_mae, lr_mae, rf_mae, hgb_mae]
})
results['MAE ($)'] = results['MAE ($)'].round(0).astype(int)
print(results.sort_values('MAE ($)'))

In [ ]:
import matplotlib.pyplot as plt

best_pred = hgb.predict(X_test)

errors = pd.DataFrame({'actual': y_test, 'abs_error': (y_test - best_pred).abs()})
errors['price_range'] = pd.cut(errors['actual'], bins=[0, 5000, 10000, 20000, 40000, 100000])
errors.groupby('price_range', observed=True)['abs_error'].mean().plot(kind='bar')
plt.ylabel('Average error ($)')
plt.title('Error by price range')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
plot_df = results.sort_values('MAE ($)', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(plot_df['Model'], plot_df['MAE ($)'])
ax.bar_label(bars, labels=[f"${v:,}" for v in plot_df['MAE ($)']], padding=4)
ax.set_xlabel('Test MAE ($), lower is better')
ax.set_title('Used car price prediction: model comparison')
ax.set_xlim(0, plot_df['MAE ($)'].max() * 1.15)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.scatter(y_test, best_pred, alpha=0.1)
plt.plot([0, 100000], [0, 100000], color='red')
plt.xlabel('Actual price'); plt.ylabel('Predicted price')
plt.title('Predicted vs actual (gradient boosting)')
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
flags = data.loc[y_test.index, ['year', 'manufacturer', 'model', 'odometer']].copy()
flags['actual'] = y_test
flags['predicted'] = best_pred.round(0)
flags['diff_pct'] = ((flags['actual'] - flags['predicted']) / flags['predicted']).round(2)

print(flags[flags['diff_pct'] > 0.25].sort_values('diff_pct', ascending=False).head(10))

In [ ]:
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 20)
print(flags[flags['diff_pct'] > 0.25].sort_values('diff_pct', ascending=False).head(10))

In [ ]:
print((flags['diff_pct'] > 0.25).sum(), "flagged overpriced")
print((flags['diff_pct'] < -0.25).sum(), "flagged deals")
print(flags[flags['diff_pct'].between(0.25, 1.0)].sample(10, random_state=1))

In [ ]:
flags['diff_usd'] = flags['actual'] - flags['predicted']
print(flags[flags['predicted'] > 3000].sort_values('diff_usd', ascending=False).head(10))